
# CMB heating: dust temperature and infrared luminosity ratio across redshift

At high redshift, the Cosmic Microwave Background (CMB) raises the dust
temperature, following da Cunha et al. (2013). Tengri applies a CMB
contrast factor ``1 - B_ν(T_CMB(z)) / B_ν(T_dust(z))`` to the emitted
spectrum, so the fraction of the absorbed luminosity that is observable
against the CMB in the 8–1000 μm rest-frame window falls for cold dust at
high redshift. The quantity plotted is that fraction at z = 10 divided by the
same fraction at z = 0.05, which isolates the CMB effect from the absolute
luminosity of the galaxy. At 25 K the ratio is ~0.35, whereas codes that
instead boost the luminosity by ``(T_z / T_0)^(4 + β)`` report ~3.9. The dust
temperature itself rises from 25 K to ~31.5 K at z = 10. Hot dust (≥ 100 K)
is unaffected because its emission sits far above the CMB at every
wavelength.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.plot import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")


def _build_model(dust_emission_dict, redshift):
    """Build a model with the specified dust_emission and redshift."""
    dust = {
        "law": "power_law",
        "type": "two_component",
        "all_params": tengri.Fixed(tengri.DEFAULT),
        "tau_diff": 0.5,
        "tau_bc": 1.0,
    }
    model = tengri.SEDModel.build(
        tengri.load_ssp(),
        sfh={"type": "const", "all_params": tengri.Fixed(tengri.DEFAULT), "log_total_mass": 11.13},
        dust_attenuation=dust,
        dust_emission=dust_emission_dict,
        redshift=tengri.Fixed(redshift),
    )
    return model


def _compute_lir_8_1000um(model):
    """
    Integrate L_IR over 8–1000 μm rest-frame.
    Returns L_IR in erg/s and the total L_ir for normalization.
    """
    state = model.predict_state({})
    wave_aa = np.asarray(state.wave)  # rest-frame Angstrom
    sed_dust_ir = np.asarray(state.derived["sed_dust_ir"])  # erg/s/Hz
    l_ir_total = float(state.derived["L_ir"])  # erg/s (total absorbed)

    # Convert wavelength to um for filtering
    wave_um = wave_aa * 1e-4

    # Speed of light in AA/s
    C_AA_S = 2.998e18
    nu = C_AA_S / wave_aa  # Hz

    # Filter to 8-1000 um
    mask = (wave_um >= 8.0) & (wave_um <= 1000.0)
    nu_masked = nu[mask]
    sed_masked = sed_dust_ir[mask]

    # Integrate L_nu over frequency (sorted ascending since nu decreases with lambda)
    order = np.argsort(nu_masked)
    l_ir_8_1000 = float(np.trapezoid(sed_masked[order], nu_masked[order]))

    return l_ir_8_1000, l_ir_total


# Temperature grid
T_grid = np.array([25.0, 30.0, 40.0, 50.0, 75.0, 100.0, 150.0, 200.0])

# Redshifts
z_low = 0.05
z_high = 10.0

# Compute ratios
ratios = []

for T in T_grid:
    # Model at low redshift (baseline)
    dust_emission = {
        "type": "casey2012",
        "beta_ir": tengri.Fixed(2.0),
        "alpha_mir": tengri.Fixed(1.4),
        "T": tengri.Fixed(T),
        "all_params": tengri.Fixed(tengri.DEFAULT),
    }
    model_low = _build_model(dust_emission, z_low)
    l_ir_low, l_abs_low = _compute_lir_8_1000um(model_low)

    # Model at high redshift (CMB heated)
    model_high = _build_model(dust_emission, z_high)
    l_ir_high, l_abs_high = _compute_lir_8_1000um(model_high)

    # Ratio of the observable 8-1000 um fraction of the absorbed luminosity.
    # Dividing each window integral by its own L_ir removes the trivial
    # dependence on how much starlight the galaxy absorbs at each redshift.
    ratio = (l_ir_high / l_abs_high) / (l_ir_low / l_abs_low)
    ratios.append(ratio)

ratios = np.array(ratios)

# Create figure
fig, ax = plt.subplots(figsize=(7.2, 5.0))

# Plot the ratio as points connected by line
ax.plot(T_grid, ratios, "o-", color="C0", lw=1.4, markersize=6, label="CMB contrast convention")

# Horizontal line at ratio = 1
ax.axhline(1.0, color="k", linestyle="--", lw=1.0, alpha=0.5, label="No CMB effect")

# Set labels and limits
ax.set(
    xlabel=r"Dust temperature at $z = 0.05$ [K]",
    ylabel=r"$(L_{8-1000}/L_{\rm abs})_{z=10} \;/\; (L_{8-1000}/L_{\rm abs})_{z=0.05}$",
    ylim=(0.2, 1.2),
)

ax.legend(frameon=False, fontsize=9)
ax.grid(True, alpha=0.3)

plt.savefig("plot_cmb_heating_lir_ratio.png", dpi=150, bbox_inches="tight")